# XAU Multi-Timeframe Setup Model

This Kaggle-ready notebook trains the first XAU setup model for the trading platform. It reads every CSV in `dataset/xau`, normalizes all timeframes into one schema, builds multi-timeframe context without future leakage, labels TP-before-SL setups, calibrates probabilities, and exports a model artifact compatible with the FastAPI backend.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, classification_report, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

try:
    from lightgbm import LGBMClassifier
    MODEL_KIND = "lightgbm"
except Exception:
    try:
        from xgboost import XGBClassifier
        MODEL_KIND = "xgboost"
    except Exception:
        from sklearn.ensemble import HistGradientBoostingClassifier
        MODEL_KIND = "sklearn_hgb"

DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/xau/xau"),
    Path("/kaggle/input/xau-dataset/xau"),
    Path("/kaggle/input/erebos-xau/xau"),
    Path("../../dataset/xau"),
    Path("dataset/xau"),
]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find XAU data folder. Update DATA_DIR_CANDIDATES for your Kaggle dataset path.")

if Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("/kaggle/working")
elif Path("dataset/xau").exists():
    OUTPUT_DIR = Path("models")
else:
    OUTPUT_DIR = Path("../../models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIMEFRAME_FILES = {
    "1m": "XAU_1m_data.csv",
    "5m": "XAU_5m_data.csv",
    "15m": "XAU_15m_data.csv",
    "30m": "XAU_30m_data.csv",
    "1h": "XAU_1h_data.csv",
    "4h": "XAU_4h_data.csv",
    "1d": "XAU_1d_data.csv",
    "1w": "XAU_1w_data.csv",
    "1M": "XAU_1Month_data.csv",
}

BASE_TIMEFRAMES = ["15m", "1h"]
CONTEXT_TIMEFRAMES = ["5m", "4h", "1d", "1w"]
SYMBOL = "XAU"
MAX_BASE_ROWS_PER_TF = None
HORIZON_BARS = {"15m": 16, "1h": 12}
SETUP_GRID = [
    {"sl_atr": 0.8, "rr": 1.5},
    {"sl_atr": 1.0, "rr": 2.0},
    {"sl_atr": 1.2, "rr": 2.5},
]

print("Data folder:", DATA_DIR)
print("Model kind:", MODEL_KIND)

## Load and Normalize CSVs

All CSVs are normalized to `symbol`, `timeframe`, `datetime`, `open`, `high`, `low`, `close`, and `volume`. The original files use semicolon separators and `YYYY.MM.DD HH:MM` timestamps.

In [ ]:
def load_timeframe(timeframe: str) -> pd.DataFrame:
    path = DATA_DIR / TIMEFRAME_FILES[timeframe]
    df = pd.read_csv(path, sep=";")
    df = df.rename(columns={
        "Date": "datetime",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df["datetime"] = pd.to_datetime(df["datetime"], format="%Y.%m.%d %H:%M", errors="coerce")
    df = df.dropna(subset=["datetime"])
    df["symbol"] = SYMBOL
    df["timeframe"] = timeframe
    numeric_cols = ["open", "high", "low", "close", "volume"]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=numeric_cols)
    df = df.drop_duplicates(subset=["datetime"], keep="last").sort_values("datetime").reset_index(drop=True)
    return df[["symbol", "timeframe", "datetime", "open", "high", "low", "close", "volume"]]

frames = {tf: load_timeframe(tf) for tf in TIMEFRAME_FILES}
for tf, df in frames.items():
    print(tf, len(df), df["datetime"].min(), "->", df["datetime"].max())

normalized_preview = pd.concat(frames.values(), ignore_index=True).head()
normalized_preview

## Feature Engineering

The model starts with practical technical context: EMA 25/99/200, RSI 14, ATR, candle anatomy, volatility, session timing, swing highs/lows, and distances to recent support/resistance/liquidity levels.

In [ ]:
def add_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).fillna(100)


def add_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    prev_close = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - prev_close).abs(),
        (df["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()


def add_swing_features(df: pd.DataFrame, lookback: int = 3) -> pd.DataFrame:
    out = df.copy()
    rolling_high = out["high"].rolling(lookback * 2 + 1, center=True).max()
    rolling_low = out["low"].rolling(lookback * 2 + 1, center=True).min()
    out["swing_high"] = np.where(out["high"].eq(rolling_high), out["high"], np.nan)
    out["swing_low"] = np.where(out["low"].eq(rolling_low), out["low"], np.nan)
    out["last_resistance"] = out["swing_high"].ffill()
    out["last_support"] = out["swing_low"].ffill()
    out["dist_to_resistance_atr"] = (out["last_resistance"] - out["close"]) / out["atr_14"]
    out["dist_to_support_atr"] = (out["close"] - out["last_support"]) / out["atr_14"]
    out["near_liquidity_high"] = (out["dist_to_resistance_atr"].between(0, 1.5)).astype(int)
    out["near_liquidity_low"] = (out["dist_to_support_atr"].between(0, 1.5)).astype(int)
    return out


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values("datetime").reset_index(drop=True)
    out["return_1"] = out["close"].pct_change()
    out["log_return_1"] = np.log(out["close"]).diff()
    out["volatility_20"] = out["log_return_1"].rolling(20).std()
    out["body_atr_raw"] = (out["close"] - out["open"]).abs()
    out["upper_wick_raw"] = out["high"] - out[["open", "close"]].max(axis=1)
    out["lower_wick_raw"] = out[["open", "close"]].min(axis=1) - out["low"]
    out["atr_14"] = add_atr(out, 14)
    for period in [25, 99, 200]:
        out[f"ema_{period}"] = out["close"].ewm(span=period, adjust=False).mean()
        out[f"ema_{period}_slope"] = out[f"ema_{period}"].diff()
        out[f"close_to_ema_{period}_atr"] = (out["close"] - out[f"ema_{period}"]) / out["atr_14"]
    out["rsi_14"] = add_rsi(out["close"], 14)
    out["body_atr"] = out["body_atr_raw"] / out["atr_14"]
    out["upper_wick_atr"] = out["upper_wick_raw"] / out["atr_14"]
    out["lower_wick_atr"] = out["lower_wick_raw"] / out["atr_14"]
    out["hour"] = out["datetime"].dt.hour
    out["dayofweek"] = out["datetime"].dt.dayofweek
    out["session_asia"] = out["hour"].between(0, 7).astype(int)
    out["session_london"] = out["hour"].between(7, 15).astype(int)
    out["session_new_york"] = out["hour"].between(13, 21).astype(int)
    out = add_swing_features(out)
    return out.replace([np.inf, -np.inf], np.nan)


feature_frames = {tf: add_features(df) for tf, df in frames.items() if tf in set(BASE_TIMEFRAMES + CONTEXT_TIMEFRAMES)}
feature_frames["15m"].tail(3)

## Multi-Timeframe Alignment

`merge_asof(..., direction="backward")` joins only the latest completed higher-timeframe candle at each base decision time. This is the key guardrail against future leakage.

In [ ]:
BASE_FEATURES = [
    "open", "high", "low", "close", "volume", "return_1", "log_return_1", "volatility_20",
    "atr_14", "rsi_14", "body_atr", "upper_wick_atr", "lower_wick_atr",
    "ema_25", "ema_99", "ema_200", "ema_25_slope", "ema_99_slope", "ema_200_slope",
    "close_to_ema_25_atr", "close_to_ema_99_atr", "close_to_ema_200_atr",
    "dist_to_resistance_atr", "dist_to_support_atr", "near_liquidity_high", "near_liquidity_low",
    "hour", "dayofweek", "session_asia", "session_london", "session_new_york",
]
CONTEXT_FEATURES = [
    "close", "atr_14", "rsi_14", "ema_25_slope", "ema_99_slope", "ema_200_slope",
    "close_to_ema_25_atr", "close_to_ema_99_atr", "close_to_ema_200_atr",
    "dist_to_resistance_atr", "dist_to_support_atr", "near_liquidity_high", "near_liquidity_low",
]


def build_base_context(base_tf: str) -> pd.DataFrame:
    base = feature_frames[base_tf][["symbol", "timeframe", "datetime"] + BASE_FEATURES].copy()
    base = base.rename(columns={col: f"base_{col}" for col in BASE_FEATURES})
    base["base_timeframe"] = base_tf
    base = base.sort_values("datetime")

    for ctx_tf in CONTEXT_TIMEFRAMES:
        if ctx_tf == base_tf or ctx_tf not in feature_frames:
            continue
        ctx_cols = ["datetime"] + CONTEXT_FEATURES
        ctx = feature_frames[ctx_tf][ctx_cols].copy().sort_values("datetime")
        ctx = ctx.rename(columns={col: f"{ctx_tf}_{col}" for col in CONTEXT_FEATURES})
        base = pd.merge_asof(base, ctx, on="datetime", direction="backward", allow_exact_matches=True)

    return base


contexts = []
for tf in BASE_TIMEFRAMES:
    ctx = build_base_context(tf)
    if MAX_BASE_ROWS_PER_TF:
        ctx = ctx.tail(MAX_BASE_ROWS_PER_TF)
    contexts.append(ctx)

decision_rows = pd.concat(contexts, ignore_index=True).sort_values("datetime").reset_index(drop=True)
print(decision_rows.shape)
decision_rows.tail(3)

## Setup Generation and TP-Before-SL Labels

Each decision candle generates long and short setups from ATR-normalized stop distance and risk/reward grid. A label is `1` if TP is touched before SL inside the horizon; `0` for SL first, timeout, or same-candle TP/SL ambiguity.

In [ ]:
def label_tp_before_sl(highs, lows, start_idx, side, stop_loss, take_profit, horizon_bars):
    end_idx = min(len(highs), start_idx + 1 + horizon_bars)
    if start_idx + 1 >= end_idx:
        return np.nan
    for idx in range(start_idx + 1, end_idx):
        if side == "long":
            hit_tp = highs[idx] >= take_profit
            hit_sl = lows[idx] <= stop_loss
        else:
            hit_tp = lows[idx] <= take_profit
            hit_sl = highs[idx] >= stop_loss
        if hit_tp and hit_sl:
            return 0
        if hit_tp:
            return 1
        if hit_sl:
            return 0
    return 0


def build_setups_for_tf(base_tf: str, context_df: pd.DataFrame) -> pd.DataFrame:
    raw = feature_frames[base_tf].reset_index(drop=True)
    highs = raw["high"].to_numpy()
    lows = raw["low"].to_numpy()
    index_by_time = pd.Series(raw.index, index=raw["datetime"]).to_dict()
    rows = []
    horizon = HORIZON_BARS[base_tf]

    for _, row in context_df.iterrows():
        idx = index_by_time.get(row["datetime"])
        atr_value = row.get("base_atr_14")
        close = row.get("base_close")
        if idx is None or not np.isfinite(atr_value) or atr_value <= 0 or not np.isfinite(close):
            continue

        for setup in SETUP_GRID:
            risk = atr_value * setup["sl_atr"]
            reward = risk * setup["rr"]
            for side in ["long", "short"]:
                if side == "long":
                    stop_loss = close - risk
                    take_profit = close + reward
                else:
                    stop_loss = close + risk
                    take_profit = close - reward
                label = label_tp_before_sl(highs, lows, idx, side, stop_loss, take_profit, horizon)
                if not np.isfinite(label):
                    continue
                payload = row.to_dict()
                payload.update({
                    "side": side,
                    "side_long": 1 if side == "long" else 0,
                    "entry": close,
                    "stop_loss": stop_loss,
                    "take_profit": take_profit,
                    "risk_reward": setup["rr"],
                    "sl_atr": setup["sl_atr"],
                    "horizon_bars": horizon,
                    "target": int(label),
                })
                rows.append(payload)
    return pd.DataFrame(rows)


setup_frames = []
for tf in BASE_TIMEFRAMES:
    ctx = decision_rows[decision_rows["base_timeframe"] == tf].copy()
    print("Building setups", tf, len(ctx))
    setup_frames.append(build_setups_for_tf(tf, ctx))

dataset = pd.concat(setup_frames, ignore_index=True).sort_values("datetime").reset_index(drop=True)
print(dataset.shape)
print(dataset["target"].value_counts(normalize=True))
dataset.tail(3)

## Time-Based Split and Training

The split is chronological: train through 2021, validate on 2022-2023, test on 2024 through the latest available XAU data. Do not use random splits for trading evaluation.

In [ ]:
DROP_COLUMNS = {
    "symbol", "timeframe", "datetime", "base_timeframe", "side", "target",
}
feature_columns = [
    col for col in dataset.columns
    if col not in DROP_COLUMNS and pd.api.types.is_numeric_dtype(dataset[col])
]

train_mask = dataset["datetime"] < "2022-01-01"
valid_mask = dataset["datetime"].between("2022-01-01", "2023-12-31 23:59:59")
test_mask = dataset["datetime"] >= "2024-01-01"

X_train, y_train = dataset.loc[train_mask, feature_columns], dataset.loc[train_mask, "target"]
X_valid, y_valid = dataset.loc[valid_mask, feature_columns], dataset.loc[valid_mask, "target"]
X_test, y_test = dataset.loc[test_mask, feature_columns], dataset.loc[test_mask, "target"]

print("Features:", len(feature_columns))
print("Train/valid/test:", X_train.shape, X_valid.shape, X_test.shape)

if MODEL_KIND == "lightgbm":
    base_model = LGBMClassifier(
        n_estimators=700,
        learning_rate=0.035,
        num_leaves=48,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="binary",
        random_state=42,
        n_jobs=-1,
    )
elif MODEL_KIND == "xgboost":
    base_model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.035,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )
else:
    base_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", HistGradientBoostingClassifier(max_iter=350, learning_rate=0.045, random_state=42)),
    ])

base_model.fit(X_train, y_train)

try:
    calibrated_model = CalibratedClassifierCV(base_model, method="isotonic", cv="prefit")
    calibrated_model.fit(X_valid, y_valid)
except TypeError:
    calibrated_model = CalibratedClassifierCV(base_model, method="sigmoid", cv="prefit")
    calibrated_model.fit(X_valid, y_valid)

calibrated_model

## Evaluation and Export

The backend expects a `joblib` bundle with `model`, `feature_columns`, and metadata. Copy the exported file to `models/xau_setup_model.joblib` in the project or set `MODEL_PATH` for the API.

In [ ]:
def evaluate(name, X, y):
    proba = calibrated_model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba) if y.nunique() > 1 else np.nan
    brier = brier_score_loss(y, proba)
    print(f"\n{name} AUC={auc:.4f} Brier={brier:.4f}")
    print(classification_report(y, pred, digits=3))
    return {"auc": float(auc), "brier": float(brier)}


metrics = {
    "valid": evaluate("valid", X_valid, y_valid),
    "test": evaluate("test", X_test, y_test),
}

bundle = {
    "model": calibrated_model,
    "feature_columns": feature_columns,
    "metadata": {
        "symbol": SYMBOL,
        "base_timeframes": BASE_TIMEFRAMES,
        "context_timeframes": CONTEXT_TIMEFRAMES,
        "target": "TP before SL inside horizon",
        "same_candle_policy": "label 0 conservatively",
        "model_kind": MODEL_KIND,
        "metrics": metrics,
        "trained_rows": int(len(X_train)),
        "valid_rows": int(len(X_valid)),
        "test_rows": int(len(X_test)),
    },
}

model_path = OUTPUT_DIR / "xau_setup_model.joblib"
config_path = OUTPUT_DIR / "xau_setup_feature_config.json"
joblib.dump(bundle, model_path)
config_path.write_text(json.dumps(bundle["metadata"] | {"feature_columns": feature_columns}, indent=2), encoding="utf-8")
print("Saved", model_path)
print("Saved", config_path)